# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesador para árboles: solo imputación por mediana (no requieren escalado)
preprocessor_tree = ColumnTransformer(
    transformers=[("num", SimpleImputer(strategy="median"), feature_columns)],
    remainder="drop",
)

# Preprocesador para modelos sensibles a la escala: imputación + StandardScaler
preprocessor_scaled = ColumnTransformer(
    transformers=[("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)],
    remainder="drop",
)

print("Preprocesadores creados para", len(feature_columns), "variables.")

Preprocesadores creados para 13 variables.


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

**Respuesta:** Porque si se ajusta con todo el conjunto de entrenamiento antes de la CV, la media y la desviación estándar se calculan usando también las filas que luego quedarán en el fold de validación. Esa información "se filtra" (*data leakage*): el fold de validación deja de ser realmente nuevo, y la métrica sale optimista y poco confiable para comparar modelos. Dentro del `Pipeline`, `cross_validate` y `GridSearchCV` llaman a `fit` solo con las filas de entrenamiento de cada fold, y las filas de validación únicamente se transforman con esos parámetros, reproduciendo lo que ocurrirá con datos nuevos en producción. Además, el pipeline completo (escalador + modelo) se guarda como un único objeto, evitando aplicar un escalado distinto en inferencia. (En este dataset el efecto probablemente sea pequeño, pero la práctica correcta es la misma.)

## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("preprocess", clone(preprocessor_tree)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("preprocess", clone(preprocessor_scaled)),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("preprocess", clone(preprocessor_scaled)),
        ("model", KNeighborsClassifier()),
    ]),
}
print(list(models))

['DecisionTree', 'LogisticRegression', 'KNN']


## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy",
                         return_train_score=True, n_jobs=1)
    rows.append({
        "model": name,
        "train_accuracy_mean": res["train_score"].mean(),
        "val_accuracy_mean": res["test_score"].mean(),
        "val_accuracy_std": res["test_score"].std(),
        "gap_train_val": res["train_score"].mean() - res["test_score"].mean(),
    })

initial_comparison = pd.DataFrame(rows).set_index("model").sort_values("val_accuracy_mean", ascending=False)
initial_comparison.round(4)

,train_accuracy_mean,val_accuracy_mean,val_accuracy_std,gap_train_val
model,,,,
LogisticRegression,1.0000,0.9791,0.0277,0.0209
KNN,0.9771,0.9507,0.0353,0.0264
DecisionTree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 2, 5],
        "model__criterion": ["gini", "entropy"],
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10, 100],   # regularización L2 (por defecto)
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    gs = GridSearchCV(pipe, param_grids[name], scoring="accuracy", cv=cv,
                      return_train_score=True, n_jobs=1, refit=True)
    gs.fit(X_train, y_train)
    searches[name] = gs
    i = gs.best_index_
    rows.append({
        "model": name,
        "best_cv_accuracy": gs.best_score_,
        "val_accuracy_std": gs.cv_results_["std_test_score"][i],
        "train_accuracy_mean": gs.cv_results_["mean_train_score"][i],
        "gap_train_val": gs.cv_results_["mean_train_score"][i] - gs.best_score_,
        "n_combinations": len(gs.cv_results_["params"]),
        "best_params": {k.replace("model__", ""): v for k, v in gs.best_params_.items()},
    })

tuned_comparison = pd.DataFrame(rows).set_index("model").sort_values("best_cv_accuracy", ascending=False)
tuned_comparison

,best_cv_accuracy,val_accuracy_std,train_accuracy_mean,gap_train_val,n_combinations,best_params
model,,,,,,
LogisticRegression,0.979064,0.027713,1.000000,0.020936,5,{'C': 1}
KNN,0.978818,0.017301,0.980624,0.001806,24,"{'n_neighbors': 7, 'p': 1, 'weights': 'uniform'}"
DecisionTree,0.902463,0.076225,0.985934,0.083471,30,"{'criterion': 'gini', 'max_depth': 5, 'min_sam..."


## 6. Selección y persistencia del campeón

In [8]:
import sklearn, datetime

# Campeón: mayor best_score_ de CV. Desempates (en este orden): menor desviación entre folds
# y menor brecha train-validación; si aún empatan, el orden de la tabla (ya ordenada por score).
ranking = tuned_comparison.reset_index().sort_values(
    ["best_cv_accuracy", "val_accuracy_std", "gap_train_val"], ascending=[False, True, True]
)
champion_name = ranking.iloc[0]["model"]
champion = searches[champion_name].best_estimator_   # pipeline completo ya reajustado con todo train
print("Campeón:", champion_name)
print(ranking[["model", "best_cv_accuracy", "val_accuracy_std", "gap_train_val"]].round(4).to_string(index=False))

# --- Persistencia ---
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# cv_results.csv: todas las combinaciones evaluadas de los tres modelos
frames = []
for name, gs in searches.items():
    r = pd.DataFrame(gs.cv_results_)
    r.insert(0, "model", name)
    r["params"] = r["params"].astype(str)
    frames.append(r[["model", "params", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]])
cv_results = pd.concat(frames, ignore_index=True)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

training_metadata = {
    "champion_model": champion_name,
    "best_params": {k.replace("model__", ""): (v if v is None or isinstance(v, (int, float, str)) else str(v))
                    for k, v in searches[champion_name].best_params_.items()},
    "cv_accuracy_mean": float(searches[champion_name].best_score_),
    "cv_accuracy_std": float(tuned_comparison.loc[champion_name, "val_accuracy_std"]),
    "cv_train_accuracy_mean": float(tuned_comparison.loc[champion_name, "train_accuracy_mean"]),
    "cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": RANDOM_STATE},
    "scoring": "accuracy",
    "all_models": {n: {"best_cv_accuracy": float(r["best_cv_accuracy"]),
                       "std": float(r["val_accuracy_std"]),
                       "best_params": {k: (v if v is None or isinstance(v, (int, float, str)) else str(v))
                                       for k, v in r["best_params"].items()}}
                   for n, r in tuned_comparison.iterrows()},
    "features": feature_columns,
    "target": contract["target"],
    "n_train_rows": int(len(X_train)),
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
    "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "test_set_used": False,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nGuardado: champion_model.joblib, training_metadata.json, cv_results.csv")

Campeón: LogisticRegression
             model  best_cv_accuracy  val_accuracy_std  gap_train_val
LogisticRegression            0.9791            0.0277         0.0209
               KNN            0.9788            0.0173         0.0018
      DecisionTree            0.9025            0.0762         0.0835

Guardado: champion_model.joblib, training_metadata.json, cv_results.csv


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

**Campeón: `LogisticRegression`** (`StandardScaler` + regresión logística con `C=1`), con accuracy de validación cruzada de **0.979** (5 folds estratificados, solo sobre train).

| Modelo | Accuracy CV (tuned) | Desv. entre folds | Brecha train–val | Combinaciones |
|---|---|---|---|---|
| LogisticRegression | **0.9791** | 0.028 | 0.021 | 5 |
| KNN (k=7, p=1, uniform) | 0.9788 | 0.017 | 0.002 | 24 |
| DecisionTree (gini, depth=5, leaf=2) | 0.9025 | 0.076 | 0.083 | 30 |

- **Desempeño:** la regresión logística y KNN quedan **prácticamente empatados** (diferencia de 0.0002 en accuracy, mucho menor que la desviación entre folds, lo que equivale a menos de una observación de 142). La decisión sigue la regla del enunciado (mayor `best_score_`), pero no hay evidencia estadística de que sea mejor que KNN. El árbol es claramente peor (≈ 7.7 puntos menos).
- **Variabilidad:** KNN es algo más estable entre folds (0.017 frente a 0.028); el árbol es el más inestable (0.076), señal de que sus resultados dependen mucho de qué filas caen en cada fold.
- **Sobreajuste:** la regresión logística llega a 1.00 en train y 0.98 en validación (brecha ≈ 0.02, aceptable); el árbol es el que más sobreajusta (brecha ≈ 0.08, pese a limitar la profundidad). KNN casi no muestra brecha. Además, con `C=0.01` (más regularizado) la logística obtiene 0.9788, o sea que el resultado es robusto frente a ese hiperparámetro.
- **Complejidad:** la logística tiene pocos parámetros (13 coeficientes por clase) y un entrenamiento y predicción muy rápidos; KNN debe guardar todos los datos de entrenamiento y calcular distancias al predecir; el árbol es simple pero inestable.
- **Interpretabilidad:** la logística permite leer el efecto de cada variable estandarizada sobre cada clase. Aunque hay multicolinealidad entre los fenoles (hallazgo del Challenge 1), la regularización L2 la controla en parte.

**Decisión:** se elige la regresión logística por tener el mejor score de CV, buena interpretabilidad, baja complejidad y una brecha pequeña. KNN queda como alternativa válida. Con solo 142 filas, la comparación definitiva y una estimación honesta del desempeño deben hacerse en el siguiente challenge sobre `test.csv`, que aquí no se tocó.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.